In [1]:
# 06_recourse_simulation.ipynb
# Worked recourse simulations: for a set of high-risk uninsured households, a DiCE
# counterfactual is generated and then "simulated" by scoring it back through the CHE
# risk model (predicted risk before vs after). A second table contrasts the single
# intervention "acquire private insurance" as seen by the predictive model versus the
# design-based (quasi-experimental) difference-in-differences estimate. Any figure is
# grayscale, no titles, 600-dpi PNG and PDF. Tables go to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from differences import ATTgt
import dice_ml

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# human-readable labels for categorical codes
INS = {"1": "NHI employee", "2": "NHI employee-dependent", "3": "NHI regional-head",
       "4": "NHI regional-member", "5": "Medical Aid-head", "6": "Medical Aid-member",
       "7": "National-merit", "8": "Uninsured(NHI)"}
ECO = {"1": "Wage worker", "2": "Public works", "3": "Employer", "4": "Self-employed",
       "5": "Unpaid family", "6": "Economically inactive"}

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
num = ["age", "hh_size", "log_hhinc"]
cat = ["female", "chronic", "htn", "dm", "disabled", "HEALTH_INS", "ECO1", "EDU", "REGION1", "hh_phi_any"]
tgt = "che25"
df = p[num + cat + [tgt, "PIDWON"]].dropna().copy(); df[tgt] = df[tgt].astype(int)
for c in cat:
    df[c] = df[c].astype(int).astype(str)

gss = GroupShuffleSplit(1, test_size=0.25, random_state=42)
tr, te = next(gss.split(df, groups=df["PIDWON"]))
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)], remainder="passthrough")
clf = Pipeline([("pre", pre), ("gb", HistGradientBoostingClassifier(
    max_iter=350, learning_rate=0.06, max_depth=4, l2_regularization=1.0, random_state=0))])
clf.fit(train[num + cat], train[tgt])

def risk(row_df):
    return float(clf.predict_proba(row_df[num + cat])[:, 1][0])
def won_to_mn(logv):
    return np.exp(logv) / 10000.0        # log-won back to units of 10,000 KRW (man-won)

# ---- six highest-risk uninsured households ----
unins = test[test["hh_phi_any"] == "0"].copy()
unins["risk"] = clf.predict_proba(unins[num + cat])[:, 1]
cases = unins.sort_values("risk", ascending=False).head(6)[num + cat].reset_index(drop=True)

# ---- generate one counterfactual per case and SIMULATE it through the model ----
dd = dice_ml.Data(dataframe=train[num + cat + [tgt]], continuous_features=num, outcome_name=tgt)
m = dice_ml.Model(model=clf, backend="sklearn")
exp = dice_ml.Dice(dd, m, method="genetic")
vary = ["hh_phi_any", "HEALTH_INS", "ECO1", "log_hhinc"]
cf = exp.generate_counterfactuals(cases, total_CFs=1, desired_class=0, features_to_vary=vary)

def fmt_change(o, c):
    parts = []
    for k in vary:
        if str(c[k]) == str(o[k]):
            continue
        if k == "hh_phi_any":
            parts.append("private insurance: No -> Yes")
        elif k == "log_hhinc":
            parts.append(f"income: {won_to_mn(float(o[k])):.0f} -> {won_to_mn(float(c[k])):.0f} man-won")
        elif k == "HEALTH_INS":
            parts.append(f"coverage: {INS.get(str(o[k]), o[k])} -> {INS.get(str(c[k]), c[k])}")
        elif k == "ECO1":
            parts.append(f"employment: {ECO.get(str(o[k]), o[k])} -> {ECO.get(str(c[k]), c[k])}")
    return "; ".join(parts) if parts else "(no change found)"

rows = []
for i in range(len(cases)):
    o = cases.iloc[i]
    r0 = risk(o.to_frame().T)
    cdf = cf.cf_examples_list[i].final_cfs_df
    if cdf is None or len(cdf) == 0:
        rows.append({"case": i + 1, "age": int(o["age"]), "income_mn": round(won_to_mn(float(o["log_hhinc"]))),
                     "coverage": INS.get(o["HEALTH_INS"]), "employment": ECO.get(o["ECO1"]),
                     "risk_orig": round(r0, 3), "recourse": "(none found)", "risk_cf": np.nan, "risk_reduction_pp": np.nan})
        continue
    c = cdf.iloc[0]
    r1 = risk(c.to_frame().T)                   # simulate the counterfactual through the model
    rows.append({"case": i + 1, "age": int(o["age"]), "income_mn": round(won_to_mn(float(o["log_hhinc"]))),
                 "coverage": INS.get(o["HEALTH_INS"]), "employment": ECO.get(o["ECO1"]),
                 "risk_orig": round(r0, 3), "recourse": fmt_change(o, c),
                 "risk_cf": round(r1, 3), "risk_reduction_pp": round(100 * (r0 - r1), 1)})
ex = pd.DataFrame(rows)
ex.to_csv(f"{TAB}/t06_recourse_examples.csv", index=False)

# ---- insurance-lever simulation: model view vs design-based (DiD) view ----
d2 = p[["PIDWON", "year", "cohort", "che25", "base_phi"]].dropna(subset=["che25"]).copy()
d2["cohort"] = d2["cohort"].replace(0, np.nan); d2 = d2.set_index(["PIDWON", "year"])
att = ATTgt(data=d2, cohort_column="cohort", base_period="universal")
att.fit(formula="che25", est_method="dr", control_group="not_yet_treated", sample_split_column="base_phi", progress_bar=False)
s = att.aggregate("simple"); s.columns = [str(c) for c in s.columns]
design_based_pp = (float(s.iloc[1, 0]) - float(s.iloc[0, 0])) * 100   # PHI minus no-PHI ATT

sim = []
for i in range(len(cases)):
    o = cases.iloc[i].copy()
    r0 = risk(o.to_frame().T)
    o2 = o.copy(); o2["hh_phi_any"] = "1"
    r1 = risk(o2.to_frame().T)                 # model-simulated effect of acquiring insurance
    sim.append({"case": i + 1, "risk_uninsured": round(r0, 3), "risk_if_insured_model": round(r1, 3),
                "model_effect_pp": round(100 * (r1 - r0), 1), "design_based_effect_pp": round(design_based_pp, 1)})
sim_tab = pd.DataFrame(sim)
sim_tab.to_csv(f"{TAB}/t06_insurance_simulation.csv", index=False)

# ---- figure: predicted risk, original vs simulated counterfactual (grayscale) ----
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(ex)); wd = 0.38
ax.bar(x - wd / 2, ex["risk_orig"], wd, color="0.25", edgecolor="0.3", label="Original")
ax.bar(x + wd / 2, ex["risk_cf"], wd, color="0.7", edgecolor="0.3", label="After DiCE recourse (simulated)")
ax.set_xticks(x); ax.set_xticklabels([f"Case {i+1}" for i in range(len(ex))])
ax.set_ylabel("Predicted P(CHE 25%)"); ax.legend(frameon=False)
save_fig(fig, "fig09_recourse_simulation")

print("Worked recourse examples:")
print(ex.to_string(index=False))
print("\nInsurance-lever simulation (model vs design-based DiD):")
print(sim_tab.to_string(index=False))
print(f"\nDesign-based (quasi-experimental) PHI effect on CHE25: {design_based_pp:.1f} pp")


  0%|          | 0/6 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:00<00:03,  1.30it/s]

 33%|███▎      | 2/6 [00:01<00:03,  1.28it/s]

 50%|█████     | 3/6 [00:02<00:02,  1.30it/s]

 67%|██████▋   | 4/6 [00:02<00:01,  1.38it/s]

 83%|████████▎ | 5/6 [00:03<00:00,  1.42it/s]

100%|██████████| 6/6 [00:04<00:00,  1.38it/s]

100%|██████████| 6/6 [00:04<00:00,  1.36it/s]

Worked recourse examples:
 case  age  income_mn               coverage            employment  risk_orig                                                                       recourse  risk_cf  risk_reduction_pp
    1   76        140 NHI employee-dependent Economically inactive      0.905 employment: Economically inactive -> Unpaid family; income: 140 -> 596 man-won    0.363               54.2
    2   77         60 NHI employee-dependent Economically inactive      0.905  coverage: NHI employee-dependent -> National-merit; income: 60 -> 441 man-won    0.451               45.5
    3   78        650 NHI employee-dependent Economically inactive      0.696                                                     income: 650 -> 889 man-won    0.546               15.0
    4   75        210 NHI employee-dependent Economically inactive      0.644                                                     income: 210 -> 441 man-won    0.428               21.6
    5   82         75    NHI regional-member Econ